In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import time
from urllib.parse import urlparse, urljoin, urldefrag

In [2]:
options = Options()
options.add_argument("--headless") 
driver = webdriver.Chrome(options=options)

In [3]:
visited_links = set()
all_links = []
pdf_links = []

In [4]:
def is_valid_url(url):
    return url.startswith("http")

In [5]:
def normalize_url(base, link):
    absolute = urljoin(base, link)
    clean_url, _ = urldefrag(absolute)
    return clean_url.rstrip("/")



def get_last_path_part(url):
    return url.strip("/").split("/")[-1]

In [ ]:
def crawl(url, same_domain_only=True):
    if url in visited_links:
        return

    print(f"Crawling: {url}")
    visited_links.add(url)

    try:
        driver.set_page_load_timeout(20) 
        driver.get(url)
        time.sleep(2)  # wait for JS to render
        html = driver.page_source
    except Exception as e:
        print(f"Failed to load {url}: {e}")
        return

    domain = urlparse(url).netloc
    last_part = get_last_path_part(url)
    with open("links.txt", "a", encoding="utf-8") as f:
        f.write(f"{url} | {last_part}\n")


    soup = BeautifulSoup(html, "html.parser")
    for a_tag in soup.find_all("a", href=True):
        href = a_tag['href']
        full_url = normalize_url(url, href)

        if not is_valid_url(full_url) or full_url in visited_links or '+971' in full_url or 'mailto:' in full_url or 'tel:' in full_url:
            continue

        if full_url.lower().endswith(".pdf"):
            with open("pdf_links.txt", "a", encoding="utf-8") as f:
                f.write(f"{url} | {last_part}\n")

        else:
            if not same_domain_only or urlparse(full_url).netloc == domain:
                crawl(full_url, same_domain_only)

In [7]:
base_url = "https://www.sharjah.ac.ae/Academics/CI/Computer-Science/"
crawl(base_url)

Crawling: https://www.sharjah.ac.ae/Academics/CI/Computer-Science/
Crawling: https://www.sharjah.ac.ae
Crawling: https://www.sharjah.ac.ae/en/Academics
Crawling: https://www.sharjah.ac.ae/en/Academics/Degree
Crawling: https://www.sharjah.ac.ae/en/Academics/Sh
Crawling: https://www.sharjah.ac.ae/en/Academics/Sh/Why-Sharia-and-Islamic-Studies
Crawling: https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff?college={9C51133F-1D30-45EB-B656-E2F273360A88}
Crawling: https://www.sharjah.ac.ae/en/Academics/ahss
Crawling: https://www.sharjah.ac.ae/en/Academics/ahss/Why-Arts-Humanities-and-Social-Studies
Crawling: https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff?college={642DBF7E-C563-48EA-B113-0B2B0F39A9C2}
Crawling: https://www.sharjah.ac.ae/en/Academics/business
Crawling: https://www.sharjah.ac.ae/en/Academics/business/Why-Business
Crawling: https://www.sharjah.ac.ae/en/Academics/Faculty-And-Staff?college={EF63801B-3D4A-4158-BBDE-CBCDE8F4A4DA}
Crawling: https://www.sharjah.ac.ae/en/

In [8]:
driver.quit()

In [9]:
print("\n✅ All Links:")
for link in all_links:
    print(link)

print("\n📄 PDF Links:")
for pdf in pdf_links:
    print(pdf)


✅ All Links:

📄 PDF Links:
